# Week 14: Attention, Transformers, Self-Supervision, BERT, and GPT


## Week 14 Lectures

- **Lecture 23:** QKV Attention and Transformers
- **Lecture 24:** Self-Supervision, BERT, and GPT


In [1]:
CHECKPOINT_DIRECTORY = __import__("pathlib").Path("checkpoints")
results = {}


## Imports and Device

The notebook uses plain PyTorch tools: `Dataset`, `DataLoader`, `nn.Embedding`, recurrent layers, optimizers, and evaluation metrics. No transformer layers are used in this week.


In [2]:
# Core Python utilities
from pathlib import Path
from collections import Counter
import os
import re
import shutil
import tarfile
import time
import urllib.request

# Numerical and plotting tools
import numpy as np
import matplotlib.pyplot as plt

# PyTorch tools
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset

# Reporting tools
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reproducibility
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

pin_memory = device.type == "cuda"
print(f"Using device: {device}")


Using device: cuda


## IMDB Sentiment Data

The main running example is IMDB sentiment classification. Each example is a sequence of words from a movie review. The label is binary:

- `0`: negative review
- `1`: positive review

The original dataset contains 25,000 training reviews and 25,000 test reviews. The notebook uses deterministic training, validation, and test splits so model comparisons use the same examples.


In [3]:
# Download and extract the IMDB dataset if needed
imdb_root = Path("../../datasets/imdb")
archive_path = imdb_root.parent / "aclImdb_v1.tar.gz"
source_url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

if not (imdb_root / "train").exists():
    imdb_root.mkdir(parents=True, exist_ok=True)
    print("Downloading IMDB dataset")
    urllib.request.urlretrieve(source_url, archive_path)

    print("Extracting IMDB dataset")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=imdb_root.parent)

    extracted_root = imdb_root.parent / "aclImdb"
    if imdb_root.exists():
        shutil.rmtree(imdb_root)
    extracted_root.rename(imdb_root)

print(f"IMDB root: {imdb_root}")


IMDB root: ../../datasets/imdb


## Tokenization and Vocabulary

Neural networks do not operate directly on strings. We convert text into token IDs:

```text
"this movie was great" -> [15, 278, 42, 91]
```

The vocabulary maps common words to integer IDs. Two special IDs are used:

- `<PAD>`: fills short reviews up to a fixed length
- `<OOV>`: represents words outside the vocabulary

This is intentionally simple tokenization. Later NLP systems use more advanced subword tokenizers, but word tokens are enough for learning RNNs and LSTMs.


In [4]:
# Simple tokenizer for IMDB reviews
TOKEN_PATTERN = re.compile(r"\b\w+\b")


def clean_text(text):
    return text.replace("<br />", " ")


def tokenize(text):
    return TOKEN_PATTERN.findall(clean_text(text).lower())


def load_imdb_split(split_root):
    split_root = Path(split_root)
    samples = []
    for class_name, label in [("neg", 0), ("pos", 1)]:
        for path in sorted((split_root / class_name).glob("*.txt")):
            text = path.read_text(encoding="utf-8", errors="ignore")
            samples.append((text, label))
    return samples


train_raw = load_imdb_split(imdb_root / "train")
test_raw = load_imdb_split(imdb_root / "test")

print(f"Raw training reviews: {len(train_raw)}")
print(f"Raw test reviews: {len(test_raw)}")


Raw training reviews: 25000
Raw test reviews: 25000


In [5]:
# Build a vocabulary from training text only
MAX_VOCAB_WORDS = 10000
SEQUENCE_LENGTH = 200


def build_vocabulary(texts, max_words=10000, min_frequency=1):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    token_to_id = {"<PAD>": 0, "<OOV>": 1}
    for word, count in counter.most_common():
        if count < min_frequency:
            continue
        if len(token_to_id) >= max_words:
            break
        token_to_id[word] = len(token_to_id)
    return token_to_id


token_to_id = build_vocabulary(
    [text for text, label in train_raw],
    max_words=MAX_VOCAB_WORDS,
)
id_to_token = {index: token for token, index in token_to_id.items()}

print(f"Vocabulary size: {len(token_to_id)}")
print("Most common learned tokens:", [id_to_token[index] for index in range(2, 12)])


Vocabulary size: 10000
Most common learned tokens: ['the', 'and', 'a', 'of', 'to', 'is', 'it', 'in', 'i', 'this']


In [6]:
# Encode one review as a fixed-length vector of token IDs
def encode_text(text, token_to_id, sequence_length):
    token_ids = [token_to_id.get(token, token_to_id["<OOV>"]) for token in tokenize(text)]
    token_ids = token_ids[:sequence_length]

    if len(token_ids) < sequence_length:
        token_ids = token_ids + [token_to_id["<PAD>"]] * (sequence_length - len(token_ids))

    return np.array(token_ids, dtype=np.int64)


example_text, example_label = train_raw[0]
example_ids = encode_text(example_text, token_to_id, SEQUENCE_LENGTH)

print("Example label:", example_label)
print("First 40 token IDs:", example_ids[:40].tolist())
print("First 40 decoded tokens:", [id_to_token.get(index, "<OOV>") for index in example_ids[:40]])


Example label: 0
First 40 token IDs: [63, 5, 4, 125, 36, 47, 7595, 1410, 16, 4, 4239, 512, 45, 17, 4, 628, 134, 12, 7, 4, 1297, 463, 5, 1740, 208, 4, 1, 7494, 303, 7, 673, 83, 35, 2140, 1100, 3023, 34, 2, 908, 1]
First 40 decoded tokens: ['story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig', 'starts', 'out', 'with', 'a', 'opening', 'scene', 'that', 'is', 'a', 'terrific', 'example', 'of', 'absurd', 'comedy', 'a', '<OOV>', 'orchestra', 'audience', 'is', 'turned', 'into', 'an', 'insane', 'violent', 'mob', 'by', 'the', 'crazy', '<OOV>']


## Dataset and DataLoaders

The `Dataset` returns one encoded review and one label. The `DataLoader` batches reviews into tensors of shape:

```text
[batch_size, sequence_length]
```

The split sizes below are part of the experiment configuration. Every model uses the same split so architecture comparisons are fair.


In [7]:
# PyTorch Dataset for IMDB sequence classification
class IMDBSequenceDataset(Dataset):
    def __init__(self, samples, token_to_id, sequence_length):
        self.samples = samples
        self.token_to_id = token_to_id
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        text, label = self.samples[index]
        token_ids = encode_text(text, self.token_to_id, self.sequence_length)
        return torch.from_numpy(token_ids), torch.tensor(label, dtype=torch.long)


TRAIN_SUBSET_SIZE = 12000
TEST_SUBSET_SIZE = 8000
BATCH_SIZE = 64
NUM_WORKERS = 2

train_dataset_full = IMDBSequenceDataset(train_raw, token_to_id, SEQUENCE_LENGTH)
test_dataset_full = IMDBSequenceDataset(test_raw, token_to_id, SEQUENCE_LENGTH)

train_indices = torch.randperm(len(train_dataset_full), generator=torch.Generator().manual_seed(SEED))[:TRAIN_SUBSET_SIZE]
test_indices = torch.randperm(len(test_dataset_full), generator=torch.Generator().manual_seed(SEED))[:TEST_SUBSET_SIZE]

train_subset = Subset(train_dataset_full, train_indices.tolist())
eval_subset = Subset(test_dataset_full, test_indices.tolist())

val_size = len(eval_subset) // 2
test_size = len(eval_subset) - val_size
val_dataset, test_dataset = random_split(
    eval_subset,
    [val_size, test_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

print(f"Training examples: {len(train_subset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Test examples: {len(test_dataset)}")


Training examples: 12000
Validation examples: 4000
Test examples: 4000


## Embeddings

An embedding layer is a lookup table. If the vocabulary has $V$ tokens and the embedding dimension is $d$, then the embedding matrix has shape:

$$V\times d$$

Each token ID selects one row of the embedding matrix. During training, the model learns token vectors that are useful for the task.


In [8]:
# Inspect embedding shapes with one batch
batch_token_ids, batch_labels = next(iter(train_loader))
embedding_layer = nn.Embedding(
    num_embeddings=len(token_to_id),
    embedding_dim=128,
    padding_idx=token_to_id["<PAD>"],
)
embedded_batch = embedding_layer(batch_token_ids)

print(f"Token ID batch shape: {tuple(batch_token_ids.shape)}")
print(f"Embedded batch shape: {tuple(embedded_batch.shape)}")
print(f"Label batch shape: {tuple(batch_labels.shape)}")


Token ID batch shape: (64, 200)
Embedded batch shape: (64, 200, 128)
Label batch shape: (64,)


## Training and Evaluation

All models use the same training function. The loss is binary cross-entropy with logits:

$$\text{BCEWithLogitsLoss}(z, y)$$

where $z$ is the raw model logit and $y \in \{0, 1\}$. Early stopping prevents wasting time after validation performance stops improving.


In [9]:
# Helpers for binary sequence classification
def binary_accuracy_from_logits(logits, labels):
    predictions = (torch.sigmoid(logits) >= 0.5).long()
    return (predictions == labels).float().mean().item()


def evaluate_binary_classifier(model, data_loader, loss_function):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for token_ids, labels in data_loader:
            token_ids = token_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(token_ids)
            loss = loss_function(logits, labels.float())

            predictions = (torch.sigmoid(logits) >= 0.5).long()
            total_loss += loss.item() * labels.size(0)
            total_correct += (predictions == labels).sum().item()
            total_seen += labels.size(0)
            all_predictions.append(predictions.cpu())
            all_labels.append(labels.cpu())

    return {
        "loss": total_loss / total_seen,
        "accuracy": total_correct / total_seen,
        "predictions": torch.cat(all_predictions),
        "labels": torch.cat(all_labels),
    }


In [10]:
# Reusable training loop for IMDB sequence models
def train_text_classifier(
    model,
    model_name,
    train_loader,
    val_loader,
    test_loader,
    *,
    max_epochs=20,
    patience=5,
    learning_rate=3e-4,
    weight_decay=1e-4,
    gradient_clip=1.0,
):
    model = model.to(device)
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    best_val_accuracy = 0.0
    best_state = None
    epochs_without_improvement = 0
    CHECKPOINT_DIRECTORY.mkdir(exist_ok=True)
    checkpoint_path = CHECKPOINT_DIRECTORY / f"best_{model_name}.pt"

    for epoch in range(max_epochs):
        start_time = time.time()
        model.train()
        total_train_loss = 0.0
        total_seen = 0

        for token_ids, labels in train_loader:
            token_ids = token_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(token_ids)
            loss = loss_function(logits, labels.float())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            optimizer.step()

            total_train_loss += loss.item() * labels.size(0)
            total_seen += labels.size(0)

        train_loss = total_train_loss / total_seen
        val_metrics = evaluate_binary_classifier(model, val_loader, loss_function)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])

        if val_metrics["accuracy"] > best_val_accuracy:
            best_val_accuracy = val_metrics["accuracy"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            torch.save(
                {
                    "model_state": best_state,
                    "history": history,
                    "best_val_accuracy": best_val_accuracy,
                },
                checkpoint_path,
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{model_name:<10} | Epoch {epoch + 1:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Acc: {val_metrics['accuracy']:.4f} | "
            f"No Improve: {epochs_without_improvement}/{patience} | "
            f"Time: {time.time() - start_time:.1f}s"
        )

        if epochs_without_improvement >= patience:
            print(f"Early stopping {model_name} at epoch {epoch + 1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_binary_classifier(model, test_loader, loss_function)
    print(f"{model_name} test accuracy: {test_metrics['accuracy']:.4f}")

    return {
        "model": model,
        "history": history,
        "best_val_accuracy": best_val_accuracy,
        "test_accuracy": test_metrics["accuracy"],
        "test_predictions": test_metrics["predictions"],
        "test_labels": test_metrics["labels"],
        "checkpoint_path": str(checkpoint_path),
    }


In [11]:
# Shared dimensions used by the transformer, MLM, and GPT-style models.
vocab_size = len(token_to_id)
pad_idx = token_to_id["<PAD>"]


# Lecture 23: QKV Attention and Transformers


## Why Simple RNNs Struggle

A vanilla RNN repeatedly applies the same kind of update:

$$h_t = 	\tanh(W_x x_t + W_h h_{t-1} + b)$$

During backpropagation through time, gradients must pass through many repeated multiplications. They can shrink toward zero or grow too large. This is one reason simple RNNs often struggle with long-range dependencies.

GRUs and LSTMs add gates. Gates let the model learn when to keep, erase, or update information. This makes it easier to preserve useful context across many timesteps.


## Transformers for Sequence Modeling

RNNs process tokens one step at a time. Transformers process a sequence with attention. Attention lets each token build a representation by looking at other tokens in the same sequence.

For a sequence of token embeddings, a transformer creates three learned projections:

$$Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V$$

The attention weights are computed from query-key similarity:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Query**: what this token is looking for
- **Key**: what each token offers for matching
- **Value**: the information each token contributes

Vaswani et al. introduced the transformer architecture using multi-head self-attention, positional encoding, feed-forward blocks, residual connections, and layer normalization. Encoder-style models such as BERT are natural for classification. Decoder-style models such as GPT are natural for next-token generation.


In [12]:
# Scaled dot-product attention from scratch
import math


def scaled_dot_product_attention(query, key, value, attention_mask=None):
    key_dimension = query.size(-1)
    scores = query @ key.transpose(-2, -1)
    scores = scores / math.sqrt(key_dimension)

    if attention_mask is not None:
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))

    attention_weights = scores.softmax(dim=-1)
    attended_values = attention_weights @ value
    return attended_values, attention_weights


# Shape demo: batch size 2, sequence length 4, feature dimension 8
example_embeddings = torch.randn(2, 4, 8)
query_projection = nn.Linear(8, 8)
key_projection = nn.Linear(8, 8)
value_projection = nn.Linear(8, 8)

queries = query_projection(example_embeddings)
keys = key_projection(example_embeddings)
values = value_projection(example_embeddings)

attended_values, attention_weights = scaled_dot_product_attention(queries, keys, values)
print(f"Attended values shape: {tuple(attended_values.shape)}")
print(f"Attention weights shape: {tuple(attention_weights.shape)}")


Attended values shape: (2, 4, 8)
Attention weights shape: (2, 4, 4)


## Transformer Encoder Classifier Trained on IMDB

The classifier below uses the same IMDB token IDs as the recurrent models. It adds positional embeddings, passes the sequence through a transformer encoder, pools the sequence representation, and predicts sentiment.

This is not BERT pretraining. It is a supervised transformer encoder trained directly for IMDB classification.


In [13]:
# Transformer encoder classifier using the notebook vocabulary
class TransformerTextClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        sequence_length,
        embed_dim=128,
        num_heads=4,
        hidden_dim=256,
        num_layers=2,
        pad_idx=0,
        dropout=0.2,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.position_embedding = nn.Embedding(sequence_length, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        positions = torch.arange(sequence_length, device=token_ids.device).unsqueeze(0)
        positions = positions.expand(batch_size, sequence_length)

        token_embeddings = self.token_embedding(token_ids)
        position_embeddings = self.position_embedding(positions)
        embeddings = token_embeddings + position_embeddings

        padding_mask = token_ids == self.pad_idx
        encoded_tokens = self.encoder(
            embeddings,
            src_key_padding_mask=padding_mask,
        )

        non_padding_mask = (token_ids != self.pad_idx).unsqueeze(-1)
        pooled = (encoded_tokens * non_padding_mask).sum(dim=1)
        pooled = pooled / non_padding_mask.sum(dim=1).clamp(min=1)
        return self.classifier(self.dropout(pooled)).squeeze(1)


transformer_classifier = TransformerTextClassifier(
    vocab_size=len(token_to_id),
    sequence_length=SEQUENCE_LENGTH,
    pad_idx=pad_idx,
)

transformer_result = train_text_classifier(
    transformer_classifier,
    "transformer_encoder",
    train_loader,
    val_loader,
    test_loader,
    max_epochs=20,
    patience=5,
    learning_rate=3e-4,
    weight_decay=1e-4,
    gradient_clip=1.0,
)
results["transformer_encoder"] = transformer_result


transformer_encoder | Epoch 001 | Train Loss: 0.6247 | Val Loss: 0.5640 | Val Acc: 0.7080 | No Improve: 0/5 | Time: 1.4s


/home/rwhite/.local/share/mamba/envs/xai-s26/lib/python3.11/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


transformer_encoder | Epoch 002 | Train Loss: 0.4804 | Val Loss: 0.5186 | Val Acc: 0.7492 | No Improve: 0/5 | Time: 1.1s
transformer_encoder | Epoch 003 | Train Loss: 0.4107 | Val Loss: 0.4775 | Val Acc: 0.7817 | No Improve: 0/5 | Time: 1.1s
transformer_encoder | Epoch 004 | Train Loss: 0.3532 | Val Loss: 0.4607 | Val Acc: 0.7857 | No Improve: 0/5 | Time: 1.1s
transformer_encoder | Epoch 005 | Train Loss: 0.3094 | Val Loss: 0.4752 | Val Acc: 0.7900 | No Improve: 0/5 | Time: 1.1s
transformer_encoder | Epoch 006 | Train Loss: 0.2653 | Val Loss: 0.5194 | Val Acc: 0.7935 | No Improve: 0/5 | Time: 1.1s
transformer_encoder | Epoch 007 | Train Loss: 0.2219 | Val Loss: 0.5660 | Val Acc: 0.7895 | No Improve: 1/5 | Time: 1.1s
transformer_encoder | Epoch 008 | Train Loss: 0.1797 | Val Loss: 0.6225 | Val Acc: 0.7915 | No Improve: 2/5 | Time: 1.1s
transformer_encoder | Epoch 009 | Train Loss: 0.1372 | Val Loss: 0.7060 | Val Acc: 0.7937 | No Improve: 0/5 | Time: 1.1s
transformer_encoder | Epoch 010 

# Lecture 24: Self-Supervision, BERT, and GPT


## Masked Language Modeling Tutorial

BERT-style pretraining uses masked language modeling. Some input tokens are replaced with a special mask token, and the model predicts the original tokens at those positions.

Example:

```text
this movie was great -> this [MASK] was great
```

The model is trained to recover `movie`. This teaches bidirectional context because the prediction can use words on both sides of the mask.


In [14]:
# Build masked-language-modeling batches from IMDB token IDs
MASK_TOKEN = "<MASK>"
mask_token_id = len(token_to_id)
mlm_vocab_size = len(token_to_id) + 1


def make_mlm_batch(token_ids, mask_probability=0.15):
    inputs = token_ids.clone()
    labels = torch.full_like(token_ids, fill_value=-100)

    can_mask = token_ids != pad_idx
    random_values = torch.rand(token_ids.shape)
    mask_positions = (random_values < mask_probability) & can_mask.cpu()

    labels[mask_positions] = token_ids[mask_positions]
    inputs[mask_positions] = mask_token_id
    return inputs, labels


class TransformerMaskedLanguageModel(nn.Module):
    def __init__(self, vocab_size, sequence_length, embed_dim=128, num_heads=4, hidden_dim=256, num_layers=2, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.position_embedding = nn.Embedding(sequence_length, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        positions = torch.arange(sequence_length, device=token_ids.device).unsqueeze(0).expand(batch_size, sequence_length)
        embeddings = self.token_embedding(token_ids) + self.position_embedding(positions)
        padding_mask = token_ids == self.pad_idx
        encoded = self.encoder(embeddings, src_key_padding_mask=padding_mask)
        return self.output_layer(encoded)


mlm_model = TransformerMaskedLanguageModel(
    vocab_size=mlm_vocab_size,
    sequence_length=SEQUENCE_LENGTH,
    pad_idx=pad_idx,
).to(device)
mlm_optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=3e-4, weight_decay=1e-4)
mlm_loss_function = nn.CrossEntropyLoss(ignore_index=-100)

MLM_EPOCHS = 20
for epoch in range(MLM_EPOCHS):
    mlm_model.train()
    total_loss = 0.0
    total_seen = 0

    for token_ids, labels in train_loader:
        masked_inputs, mlm_labels = make_mlm_batch(token_ids)
        masked_inputs = masked_inputs.to(device, non_blocking=True)
        mlm_labels = mlm_labels.to(device, non_blocking=True)

        token_logits = mlm_model(masked_inputs)
        loss = mlm_loss_function(
            token_logits.reshape(-1, mlm_vocab_size),
            mlm_labels.reshape(-1),
        )

        mlm_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), max_norm=1.0)
        mlm_optimizer.step()

        total_loss += loss.item() * masked_inputs.size(0)
        total_seen += masked_inputs.size(0)

    print(f"MLM epoch {epoch + 1:03d} | Loss: {total_loss / total_seen:.4f}")


MLM epoch 001 | Loss: 7.0075
MLM epoch 002 | Loss: 6.4601
MLM epoch 003 | Loss: 6.4342
MLM epoch 004 | Loss: 6.4342
MLM epoch 005 | Loss: 6.4147
MLM epoch 006 | Loss: 6.4166
MLM epoch 007 | Loss: 6.4158
MLM epoch 008 | Loss: 6.4006
MLM epoch 009 | Loss: 6.4008
MLM epoch 010 | Loss: 6.3934
MLM epoch 011 | Loss: 6.3801
MLM epoch 012 | Loss: 6.3778
MLM epoch 013 | Loss: 6.3745
MLM epoch 014 | Loss: 6.3661
MLM epoch 015 | Loss: 6.3630
MLM epoch 016 | Loss: 6.3538
MLM epoch 017 | Loss: 6.3549
MLM epoch 018 | Loss: 6.3467
MLM epoch 019 | Loss: 6.3348
MLM epoch 020 | Loss: 6.3338


## Fine-Tuning the MLM Encoder on IMDB

After MLM pretraining, the encoder has learned token representations from unlabeled text. To fine-tune it for sentiment classification, attach a classification head and train with sentiment labels.


In [15]:
# Fine-tune the MLM encoder for IMDB sentiment classification
class MLMEncoderClassifier(nn.Module):
    def __init__(self, mlm_model, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.token_embedding = mlm_model.token_embedding
        self.position_embedding = mlm_model.position_embedding
        self.encoder = mlm_model.encoder
        embed_dim = mlm_model.output_layer.in_features
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        positions = torch.arange(sequence_length, device=token_ids.device).unsqueeze(0).expand(batch_size, sequence_length)
        embeddings = self.token_embedding(token_ids) + self.position_embedding(positions)
        padding_mask = token_ids == self.pad_idx
        encoded = self.encoder(embeddings, src_key_padding_mask=padding_mask)
        non_padding_mask = (token_ids != self.pad_idx).unsqueeze(-1)
        pooled = (encoded * non_padding_mask).sum(dim=1) / non_padding_mask.sum(dim=1).clamp(min=1)
        return self.classifier(pooled).squeeze(1)


mlm_finetuned_classifier = MLMEncoderClassifier(mlm_model, pad_idx=pad_idx)
mlm_finetuned_result = train_text_classifier(
    mlm_finetuned_classifier,
    "mlm_encoder_finetuned",
    train_loader,
    val_loader,
    test_loader,
    max_epochs=100,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    gradient_clip=1.0,
)
results["mlm_encoder_finetuned"] = mlm_finetuned_result


mlm_encoder_finetuned | Epoch 001 | Train Loss: 0.5633 | Val Loss: 0.5398 | Val Acc: 0.7312 | No Improve: 0/5 | Time: 1.2s
mlm_encoder_finetuned | Epoch 002 | Train Loss: 0.4731 | Val Loss: 0.4916 | Val Acc: 0.7630 | No Improve: 0/5 | Time: 1.2s
mlm_encoder_finetuned | Epoch 003 | Train Loss: 0.4229 | Val Loss: 0.4757 | Val Acc: 0.7725 | No Improve: 0/5 | Time: 1.2s
mlm_encoder_finetuned | Epoch 004 | Train Loss: 0.3855 | Val Loss: 0.4751 | Val Acc: 0.7823 | No Improve: 0/5 | Time: 1.2s
mlm_encoder_finetuned | Epoch 005 | Train Loss: 0.3498 | Val Loss: 0.4781 | Val Acc: 0.7795 | No Improve: 1/5 | Time: 1.2s
mlm_encoder_finetuned | Epoch 006 | Train Loss: 0.3174 | Val Loss: 0.4811 | Val Acc: 0.7835 | No Improve: 0/5 | Time: 1.2s
mlm_encoder_finetuned | Epoch 007 | Train Loss: 0.2755 | Val Loss: 0.5170 | Val Acc: 0.7843 | No Improve: 0/5 | Time: 1.3s
mlm_encoder_finetuned | Epoch 008 | Train Loss: 0.2355 | Val Loss: 0.6018 | Val Acc: 0.7738 | No Improve: 1/5 | Time: 1.2s
mlm_encoder_fine

## GPT-Style Causal Language Modeling Tutorial

GPT-style pretraining predicts the next token from previous tokens. The model uses a causal mask so position $t$ cannot attend to future positions.

```text
input:  this movie was
label:       movie was great
```

This objective trains a left-to-right language model. For classification, a GPT-style model can use the final non-padding token representation as the sequence summary.


In [16]:
# GPT-style decoder-only transformer for causal language modeling
class CausalTransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, sequence_length, embed_dim=128, num_heads=4, hidden_dim=256, num_layers=2, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.sequence_length = sequence_length
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.position_embedding = nn.Embedding(sequence_length, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True,
            activation="gelu",
        )
        self.decoder_blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        positions = torch.arange(sequence_length, device=token_ids.device).unsqueeze(0).expand(batch_size, sequence_length)
        embeddings = self.token_embedding(token_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, device=token_ids.device),
            diagonal=1,
        ).bool()
        padding_mask = token_ids == self.pad_idx
        hidden_states = self.decoder_blocks(
            embeddings,
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )
        return self.output_layer(hidden_states), hidden_states


gpt_vocab_size = len(token_to_id)
gpt_model = CausalTransformerLanguageModel(
    vocab_size=gpt_vocab_size,
    sequence_length=SEQUENCE_LENGTH,
    pad_idx=pad_idx,
).to(device)
gpt_optimizer = torch.optim.AdamW(gpt_model.parameters(), lr=3e-4, weight_decay=1e-4)
gpt_loss_function = nn.CrossEntropyLoss(ignore_index=pad_idx)

GPT_PRETRAIN_EPOCHS = 20
for epoch in range(GPT_PRETRAIN_EPOCHS):
    gpt_model.train()
    total_loss = 0.0
    total_seen = 0

    for token_ids, labels in train_loader:
        input_ids = token_ids[:, :-1].to(device, non_blocking=True)
        target_ids = token_ids[:, 1:].to(device, non_blocking=True)

        token_logits, hidden_states = gpt_model(input_ids)
        loss = gpt_loss_function(
            token_logits.reshape(-1, gpt_vocab_size),
            target_ids.reshape(-1),
        )

        gpt_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt_model.parameters(), max_norm=1.0)
        gpt_optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        total_seen += input_ids.size(0)

    print(f"GPT-style LM epoch {epoch + 1:03d} | Loss: {total_loss / total_seen:.4f}")


GPT-style LM epoch 001 | Loss: 6.9814
GPT-style LM epoch 002 | Loss: 6.2124
GPT-style LM epoch 003 | Loss: 5.9634
GPT-style LM epoch 004 | Loss: 5.8186
GPT-style LM epoch 005 | Loss: 5.7244
GPT-style LM epoch 006 | Loss: 5.6526
GPT-style LM epoch 007 | Loss: 5.5928
GPT-style LM epoch 008 | Loss: 5.5408
GPT-style LM epoch 009 | Loss: 5.4938
GPT-style LM epoch 010 | Loss: 5.4521
GPT-style LM epoch 011 | Loss: 5.4127
GPT-style LM epoch 012 | Loss: 5.3771
GPT-style LM epoch 013 | Loss: 5.3438
GPT-style LM epoch 014 | Loss: 5.3120
GPT-style LM epoch 015 | Loss: 5.2824
GPT-style LM epoch 016 | Loss: 5.2551
GPT-style LM epoch 017 | Loss: 5.2281
GPT-style LM epoch 018 | Loss: 5.2026
GPT-style LM epoch 019 | Loss: 5.1789
GPT-style LM epoch 020 | Loss: 5.1561


In [17]:
# Fine-tune the GPT-style model on IMDB sentiment
class CausalTransformerClassifier(nn.Module):
    def __init__(self, language_model, pad_idx=0):
        super().__init__()
        self.language_model = language_model
        self.pad_idx = pad_idx
        embed_dim = language_model.output_layer.in_features
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, token_ids):
        token_logits, hidden_states = self.language_model(token_ids)
        non_padding_lengths = (token_ids != self.pad_idx).sum(dim=1).clamp(min=1)
        final_indices = non_padding_lengths - 1
        batch_indices = torch.arange(token_ids.size(0), device=token_ids.device)
        final_hidden = hidden_states[batch_indices, final_indices]
        return self.classifier(final_hidden).squeeze(1)


gpt_finetuned_classifier = CausalTransformerClassifier(gpt_model, pad_idx=pad_idx)
gpt_finetuned_result = train_text_classifier(
    gpt_finetuned_classifier,
    "gpt_style_finetuned",
    train_loader,
    val_loader,
    test_loader,
    max_epochs=100,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    gradient_clip=1.0,
)
results["gpt_style_finetuned"] = gpt_finetuned_result


gpt_style_finetuned | Epoch 001 | Train Loss: 0.6827 | Val Loss: 0.6286 | Val Acc: 0.6472 | No Improve: 0/5 | Time: 1.4s
gpt_style_finetuned | Epoch 002 | Train Loss: 0.5688 | Val Loss: 0.5642 | Val Acc: 0.7100 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 003 | Train Loss: 0.4865 | Val Loss: 0.5301 | Val Acc: 0.7368 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 004 | Train Loss: 0.4217 | Val Loss: 0.5111 | Val Acc: 0.7575 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 005 | Train Loss: 0.3656 | Val Loss: 0.5194 | Val Acc: 0.7590 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 006 | Train Loss: 0.3132 | Val Loss: 0.5473 | Val Acc: 0.7612 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 007 | Train Loss: 0.2736 | Val Loss: 0.5738 | Val Acc: 0.7620 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 008 | Train Loss: 0.2227 | Val Loss: 0.6275 | Val Acc: 0.7635 | No Improve: 0/5 | Time: 1.3s
gpt_style_finetuned | Epoch 009 

## Off-the-Shelf Pretrained Transformers

The earlier transformer models were trained inside this notebook. Modern practice often starts from pretrained models.

- **BERT-style** models are encoder-only and are commonly fine-tuned for classification.
- **GPT-style** models are decoder-only and can also be fine-tuned for classification by adding a classification head.

The cells below use Hugging Face `transformers`. This package is required for the pretrained-model section.


In [18]:
# Hugging Face tools for pretrained transformer fine-tuning
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification


In [19]:
# Dataset wrapper that tokenizes raw IMDB text with a pretrained tokenizer
class HuggingFaceIMDBDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=256):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        text, label = self.samples[index]
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


def balanced_sample(samples, sample_size, seed=0):
    negative_samples = [sample for sample in samples if sample[1] == 0]
    positive_samples = [sample for sample in samples if sample[1] == 1]

    generator = torch.Generator().manual_seed(seed)
    per_class = sample_size // 2
    negative_indices = torch.randperm(len(negative_samples), generator=generator)[:per_class]
    positive_indices = torch.randperm(len(positive_samples), generator=generator)[:per_class]

    sampled = [negative_samples[index] for index in negative_indices.tolist()]
    sampled += [positive_samples[index] for index in positive_indices.tolist()]

    permutation = torch.randperm(len(sampled), generator=generator)
    return [sampled[index] for index in permutation.tolist()]


def make_hf_loaders(tokenizer, train_size=2000, eval_size=1000, batch_size=8, max_length=256):
    train_samples = balanced_sample(train_raw, train_size, seed=SEED)
    eval_samples = balanced_sample(test_raw, eval_size, seed=SEED + 1)

    split = eval_size // 2
    val_samples = eval_samples[:split]
    test_samples = eval_samples[split:]

    print("HF train labels:", Counter(label for text, label in train_samples))
    print("HF val labels:", Counter(label for text, label in val_samples))
    print("HF test labels:", Counter(label for text, label in test_samples))

    train_dataset = HuggingFaceIMDBDataset(train_samples, tokenizer, max_length=max_length)
    val_dataset = HuggingFaceIMDBDataset(val_samples, tokenizer, max_length=max_length)
    test_dataset = HuggingFaceIMDBDataset(test_samples, tokenizer, max_length=max_length)

    train_loader_hf = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
    val_loader_hf = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    test_loader_hf = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    return train_loader_hf, val_loader_hf, test_loader_hf


In [20]:
# Fine-tune a pretrained transformer sequence classifier
def fine_tune_pretrained_sequence_classifier(
    model_name,
    run_name,
    *,
    max_epochs=2,
    patience=2,
    learning_rate=2e-5,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_loader_hf, val_loader_hf, test_loader_hf = make_hf_loaders(tokenizer)

    label2id = {"negative": 0, "positive": 1}
    id2label = {0: "negative", 1: "positive"}
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=2,
        label2id=label2id,
        id2label=id2label,
        pad_token_id=tokenizer.pad_token_id,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
    best_val_accuracy = 0.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        total_loss = 0.0
        seen = 0

        for batch in train_loader_hf:
            batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * batch["labels"].size(0)
            seen += batch["labels"].size(0)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader_hf:
                batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
                logits = model(**batch).logits
                predictions = logits.argmax(dim=1)
                correct += (predictions == batch["labels"]).sum().item()
                total += batch["labels"].size(0)

        val_accuracy = correct / total
        print(
            f"{run_name:<16} | Epoch {epoch + 1:03d} | "
            f"Train Loss: {total_loss / seen:.4f} | Val Acc: {val_accuracy:.4f}"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping {run_name} at epoch {epoch + 1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader_hf:
            batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
            logits = model(**batch).logits
            all_predictions.append(logits.argmax(dim=1).cpu())
            all_labels.append(batch["labels"].cpu())

    predictions = torch.cat(all_predictions)
    labels = torch.cat(all_labels)
    test_accuracy = (predictions == labels).float().mean().item()
    print(f"{run_name} test accuracy: {test_accuracy:.4f}")

    return {
        "model": model,
        "tokenizer": tokenizer,
        "best_val_accuracy": best_val_accuracy,
        "test_accuracy": test_accuracy,
        "test_predictions": predictions,
        "test_labels": labels,
    }


In [21]:
# Fine-tune off-the-shelf BERT- and GPT-style models on IMDB
bert_finetuned_result = fine_tune_pretrained_sequence_classifier(
    "distilbert-base-uncased",
    "distilbert_imdb",
    max_epochs=100,
    patience=5,
    learning_rate=2e-5,
)

gpt_finetuned_result = fine_tune_pretrained_sequence_classifier(
    "distilgpt2",
    "distilgpt2_imdb",
    max_epochs=100,
    patience=5,
    learning_rate=2e-5,
)


HF train labels: Counter({1: 1000, 0: 1000})
HF val labels: Counter({1: 256, 0: 244})
HF test labels: Counter({0: 256, 1: 244})


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distilbert_imdb  | Epoch 001 | Train Loss: 0.4585 | Val Acc: 0.8260
distilbert_imdb  | Epoch 002 | Train Loss: 0.2704 | Val Acc: 0.8720
distilbert_imdb  | Epoch 003 | Train Loss: 0.1622 | Val Acc: 0.8680
distilbert_imdb  | Epoch 004 | Train Loss: 0.0881 | Val Acc: 0.8800
distilbert_imdb  | Epoch 005 | Train Loss: 0.0620 | Val Acc: 0.8380
distilbert_imdb  | Epoch 006 | Train Loss: 0.0461 | Val Acc: 0.8620
distilbert_imdb  | Epoch 007 | Train Loss: 0.0303 | Val Acc: 0.8800
distilbert_imdb  | Epoch 008 | Train Loss: 0.0351 | Val Acc: 0.8820
distilbert_imdb  | Epoch 009 | Train Loss: 0.0143 | Val Acc: 0.8700
distilbert_imdb  | Epoch 010 | Train Loss: 0.0272 | Val Acc: 0.8740
distilbert_imdb  | Epoch 011 | Train Loss: 0.0081 | Val Acc: 0.8880
distilbert_imdb  | Epoch 012 | Train Loss: 0.0058 | Val Acc: 0.8700
distilbert_imdb  | Epoch 013 | Train Loss: 0.0077 | Val Acc: 0.8740
distilbert_imdb  | Epoch 014 | Train Loss: 0.0244 | Val Acc: 0.8800
distilbert_imdb  | Epoch 015 | Train Loss: 0.049

distilbert_imdb test accuracy: 0.8680


You passed `num_labels=2` which is incompatible to the `id2label` map of length `1`.


HF train labels: Counter({1: 1000, 0: 1000})
HF val labels: Counter({1: 256, 0: 244})
HF test labels: Counter({0: 256, 1: 244})


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distilgpt2_imdb  | Epoch 001 | Train Loss: 0.5642 | Val Acc: 0.8120
distilgpt2_imdb  | Epoch 002 | Train Loss: 0.3465 | Val Acc: 0.8460
distilgpt2_imdb  | Epoch 003 | Train Loss: 0.2858 | Val Acc: 0.8500
distilgpt2_imdb  | Epoch 004 | Train Loss: 0.2289 | Val Acc: 0.8560
distilgpt2_imdb  | Epoch 005 | Train Loss: 0.1648 | Val Acc: 0.8720
distilgpt2_imdb  | Epoch 006 | Train Loss: 0.1132 | Val Acc: 0.8660
distilgpt2_imdb  | Epoch 007 | Train Loss: 0.0876 | Val Acc: 0.8540
distilgpt2_imdb  | Epoch 008 | Train Loss: 0.0690 | Val Acc: 0.8680
distilgpt2_imdb  | Epoch 009 | Train Loss: 0.0503 | Val Acc: 0.8560
distilgpt2_imdb  | Epoch 010 | Train Loss: 0.0422 | Val Acc: 0.8600
Early stopping distilgpt2_imdb at epoch 10
distilgpt2_imdb test accuracy: 0.8660


## Takeaways

Sequence models process ordered data. A recurrent model maintains a hidden state that changes as it reads the sequence.

Mean-pooled embeddings are fast and useful, but they mostly ignore order. CNNs capture local word patterns. Simple RNNs introduce sequential memory, while GRUs and LSTMs improve memory with gates. Bidirectional LSTMs use both left and right context when the whole sequence is available.

Transformers use attention instead of recurrence. Query-key-value attention lets each token decide which other tokens matter. Encoder-style transformers such as BERT are natural for classification and masked language modeling. Decoder-style transformers such as GPT are natural for causal language modeling and generation.

For IMDB sentiment classification, model quality depends on architecture, data size, sequence length, vocabulary, optimization, and pretraining. The important habit is to compare models under the same data split and training procedure.
